In [1]:
!pip3 install --upgrade --quiet langchain langchain-community langchain-openai chromadb
!pip3 install --upgrade --quiet pypdf pandas streamlit python-dotenv
!pip3 install --upgrade --quiet langchain-text-splitters
!pip3 install --upgrade --quiet langchain-groq
!pip3 install --upgrade --quiet langchain-huggingface sentence-transformers


In [2]:
# Import Langchain modules
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_chroma import Chroma
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from langchain_huggingface import HuggingFaceEmbeddings



# Other modules and packages
import os
import tempfile
import streamlit as st
import pandas as pd
from dotenv import load_dotenv

/var/folders/qq/8vrms0tn15lf8xcmrcq_wbxc0000gp/T/ipykernel_92639/4136494175.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/Users/rushikeshchopade/Documents/RAGLLMs/myenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
load_dotenv()  # Load environment variables from .env file

True

In [4]:
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

### Define Our LLM


In [5]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.1,
)
llm.invoke("Tell me a dad joke")

AIMessage(content="Here's one: Why did the mushroom go to the party? Because he was a fun-gi.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 40, 'total_tokens': 62, 'completion_time': 0.047322731, 'completion_tokens_details': None, 'prompt_time': 0.003696564, 'prompt_tokens_details': None, 'queue_time': 0.394597994, 'total_time': 0.051019295}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_ce7bc1685b', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e5a71-b559-7ce1-98ce-26dbde55196f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 40, 'output_tokens': 22, 'total_tokens': 62})

### Process PDF Document


In [6]:
import glob

# Load all PDF files from data folder
pdf_files = glob.glob("data/*.pdf")
print(f"Found {len(pdf_files)} PDF files: {pdf_files}")

pages = []
for pdf_file in pdf_files:
    loader = PyPDFLoader(pdf_file)
    pages.extend(loader.load())

print(f"Total pages loaded: {len(pages)}")
pages

Found 3 PDF files: ['data/Frontend_QnA2.pdf', 'data/70-javascript-interview-questions-5gfi.pdf', 'data/Frontend_QnA1.pdf']
Total pages loaded: 108


[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': 'data/Frontend_QnA2.pdf', 'total_pages': 23, 'page': 0, 'page_label': '1'}, page_content='Inter view Questions\nFront -End\nwit h Answ ers t o Cr ack\nTechnic al Inter view'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': 'data/Frontend_QnA2.pdf', 'total_pages': 23, 'page': 1, 'page_label': '2'}, page_content='P r e p a r e  f o r  y o u r  F r o n t - e n d  I n t e r v i e w  w i t h  t h e s e  \nm o s t  a s k e d  i n t e r v i e w  q u e s t i o n s . \r\n\r\nM a k s e  s u r e  t o  h a v e  a  t h o r o u g h  k n o w l e d g e  i n t o  \nk e y  f r o n t - e n d  t e c h n o l o g i e s ,  i n c l u d i n g  H T M L ,  \nC S S ,  J a v a S c r i p t ,  a n d  p o p u l a r  f r a m e w o r k s /\nl i b r a r i e s  l i k e  R e a c t ,  V u e ,  o r  A n g u l a r .\n*Disc laimer *\nwww .bosscoderacadem y .com 1'),
 Document(metadata={'produc

### Split the Documents


In [7]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    separators=["\n\n", "\n", " ", ""],
)
chunks = text_splitter.split_documents(pages)

### Create Embeddings


In [8]:
def get_embedding_function():
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
    return embeddings


embedding_function = get_embedding_function()
test_vector = embedding_function.embed_query("car")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9272.46it/s]


In [9]:
from langchain_classic.evaluation import load_evaluator

evaluator = load_evaluator("embedding_distance", embeddings=embedding_function)

evaluator.evaluate_strings(prediction="car", reference="automobile")

{'score': 0.1354718143371938}

In [10]:
evaluator.evaluate_strings(prediction="India", reference="Developing country")

{'score': 0.4781390354424624}

In [11]:
%pip install langchain-chroma

Note: you may need to restart the kernel to use updated packages.


### Create vector database


In [12]:
import uuid


def create_vectorstore(
    chunks, embedding_function, vectorstore_path="vectorstore_chroma"
):

    # create list of unique ids for each chunk
    ids = [str(uuid.uuid5(uuid.NAMESPACE_DNS, chunk.page_content)) for chunk in chunks]

    # Ensure the unique IDs are assigned to the corresponding chunks

    unique_ids = set()
    unique_chunks = []

    for chunk, id in zip(chunks, ids):
        if id not in unique_ids:
            unique_ids.add(id)
            chunk.metadata["id"] = id  # Assign the unique ID to the chunk's metadata
            unique_chunks.append(chunk)

    vectorstore = Chroma.from_documents(
        documents=unique_chunks,
        embedding=embedding_function,
        persist_directory=vectorstore_path,
    )

    return vectorstore

In [13]:
vectorstore = create_vectorstore(
    chunks, embedding_function, vectorstore_path="vectorstore_chroma"
)

### Query for relevant data


In [14]:
# Load vectorstore from disk
vectorstore = Chroma(
    persist_directory="vectorstore_chroma", embedding_function=embedding_function
)

In [28]:
# Create retriever and get relevant chunks

retriever = vectorstore.as_retriever(search_type="similarity")
relevant_chunks = retriever.invoke("What are the ways of making objects in JavaScript?")
relevant_chunks

[Document(id='b2e5c827-8554-421b-a9b3-a24085bf572d', metadata={'moddate': '2026-05-24T14:41:31+00:00', 'total_pages': 68, 'page_label': '57', 'id': 'ca4e3fb6-05ab-5f17-a4c4-9e3a34e4ba29', 'producer': 'PDF Candy', 'source': 'data/70-javascript-interview-questions-5gfi.pdf', 'page': 56, 'creator': 'PDF Candy', 'creationdate': '2026-05-24T14:41:31+00:00'}, page_content='Technologies use for AJAX.\nHTML - web page structure\nCSS - the styling for the webpage\nJavaScript - the behaviour of the webpage and updates to the DOM\nXMLHttpRequest API - used to send and retrieve data from the server\nPHP,Python,Nodejs - Some Server-Side language\n61. What are the ways of making objects in JavaScript?\n↑ Using Object Literal. \nconst o = {  \n  "prop" : "bwahahah", \n  "prop2" : "hweasa" \n}; \nconsole.log("prop" in o); //This logs true indicating the property "prop" is in "o" obje\nconsole.log("prop1" in o); //This logs false indicating the property "prop" is not in  \n//Still using the o object in

In [25]:
# Prompt template for question answering

PROMPT_TEMPLATE = """
You are a helpful assistant for answering questions about frontend development. Use the following retrieved chunks to answer the question. If you don't know the answer, say you don't know. DON'T MAKE UP ANYTHING.

{context}

---

Answer the question based on the above context: {question}
"""

In [26]:
# Concentrate context text
context_text = "\n\n".join([chunk.page_content for chunk in relevant_chunks])

# Create prompt
prompt_template = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)
propmt = prompt_template.format(
    context=context_text, question="What are the ways of making objects in JavaScript?"
)
print(propmt)

Human: 
You are a helpful assistant for answering questions about frontend development. Use the following retrieved chunks to answer the question. If you don't know the answer, say you don't know. DON'T MAKE UP ANYTHING.

Technologies use for AJAX.
HTML - web page structure
CSS - the styling for the webpage
JavaScript - the behaviour of the webpage and updates to the DOM
XMLHttpRequest API - used to send and retrieve data from the server
PHP,Python,Nodejs - Some Server-Side language
61. What are the ways of making objects in JavaScript?
↑ Using Object Literal. 
const o = {  
  "prop" : "bwahahah", 
  "prop2" : "hweasa" 
}; 
console.log("prop" in o); //This logs true indicating the property "prop" is in "o" obje
console.log("prop1" in o); //This logs false indicating the property "prop" is not in  
//Still using the o object in the first example. 
console.log(o.hasOwnProperty("prop2")); // This logs true 
console.log(o.hasOwnProperty("prop1")); // This logs false 
//Still using the o ob

# Generate responses


In [27]:
llm.invoke(propmt)

AIMessage(content='In JavaScript, there are several ways to create objects. Based on the provided information, the following methods are mentioned:\n\n1. **Using Object Literal**: This involves creating an object using curly brackets `{}` and defining properties and values inside. For example:\n   ```\n   const o = { \n     "prop" : "bwahahah", \n     "prop2" : "hweasa" \n   };\n   ```\n2. **Using the `Object.create()` method**: This method allows you to create a new object with a specified prototype. For example:\n   ```\n   const n = { \n     greeting() { \n       return `Hi, I\'m ${this.name}`; \n     } \n   };\n   const o = Object.create(n); // sets the prototype of "o" to be "n"\n   o.name = "Mark";\n   ```\n3. **Using a constructor function**: Although not explicitly shown in the provided text, it\'s implied through the mention of `const mark = new Person("Mark");`, suggesting that you can create objects using constructor functions and the `new` keyword.\n\nThese are the methods 

### Using Langchain Expression Langauge


In [29]:
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt_template
    | llm
)
rag_chain.invoke("What are the ways of making objects in JavaScript?")

AIMessage(content='The ways of making objects in JavaScript are:\n\n1. Using Object Literal: \n   Example: `const o = { "prop" : "bwahahah", "prop2" : "hweasa" };`\n\n2. Using Constructor Functions: \n   Example: \n   ```\n   function Person(name) { \n      this.name = name; \n   } \n   Person.prototype.greeting = function () { \n      return `Hi, I\'m ${this.name}`; \n   } \n   const mark = new Person("Mark"); \n   ```\n\n3. Using Object.create method: \n   Example: \n   ```\n   const n = { \n      greeting() { \n         return `Hi, I\'m ${this.name}`; \n      } \n   }; \n   const o = Object.create(n); \n   o.name = "Mark"; \n   ```', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 179, 'prompt_tokens': 932, 'total_tokens': 1111, 'completion_time': 0.297509374, 'completion_tokens_details': None, 'prompt_time': 0.076093955, 'prompt_tokens_details': None, 'queue_time': 0.158720444, 'total_time': 0.373603329}, 'model_name': 'llama-3.3-70b-versatile', 'syste

### Generate structured responses

In [30]:
class ExtractedInfo(BaseModel):
    paper_title: str
    paper_summary: str
    paper_authors: str
    answer: str
    additional_info: str


In [32]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt_template
    | llm.with_structured_output(ExtractedInfo)
)

rag_chain.invoke("What are Closures?")

ExtractedInfo(paper_title='Closures', paper_summary="Closures is simply the ability of a function at the time of declaration to remember the references of variables and parameters on its current scope, on its parent function scope, on its parent's parent function scope until it reaches the global scope with the help of Scope Chain", paper_authors='Unknown', answer="Closures is the ability of a function to remember the references of variables and parameters on its current scope, on its parent function scope, on its parent's parent function scope until it reaches the global scope with the help of Scope Chain", additional_info='Closures can be tricky to understand and are often demonstrated with examples, such as the one provided in the text where a function remembers the value of a variable from its parent scope')

### Transform response into a dataframe

In [33]:
structured_response = rag_chain.invoke(
    "What are Closures?"
)
df = pd.DataFrame([structured_response.model_dump()])

# Transforming into a table with two rows: answer and additional info

answer_row = []

for col in df.columns:
    answer_row.append(df[col][0])

# Create a new DataFrame with two rows: answer and additional info
structured_response_df = pd.DataFrame(
    [answer_row],
    columns=df.columns,
    index=["Answer"],
)
structured_response_df

,paper_title,paper_summary,paper_authors,answer,additional_info
Answer,Closures,Closures is simply the ability of a function a...,Unknown,Closures is the ability of a function to remem...,Closures can be tricky to understand and are o...
